# 技能0 · Day 5 上机：数据治理与 SQL · 电商营销数据库

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **sqlite3**（Python 内置库）创建电商营销数据库 Schema（categories/customers/products/orders/order_items/campaigns 六表），理解主键/外键/CHECK/DEFAULT/GENERATED 约束
2. 用 **SQL DQL**（SELECT/WHERE/JOIN/GROUP BY/HAVING/窗口函数/子查询）完成营销分析，用 `pandas.read_sql()` 将结果转 DataFrame
3. 执行数据治理实操：创建索引、检测缺失值与重复记录、评估数据质量六维度
4. 理解范式化（1NF/2NF/3NF）原则，并在 Schema 设计中嵌入合规要求

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：sqlite3（Python 标准库，零安装）+ pandas.read_sql（SQL 结果转 DataFrame）。
营销映射：6 张表（类目/客户/商品/订单/订单明细/营销活动），用 SQL 完成从 Schema 设计到营销分析的完整闭环。


## 0. 环境准备

> sqlite3 是 Python 标准库，无需安装。pandas 需 `pip install pandas`。
> pandas 2.x 推荐：基于 Apache Arrow 的新后端可提升内存效率 30-50%。


In [ ]:
# 首次运行需安装 pandas（sqlite3 是标准库无需安装）
# !pip install pandas numpy -q


## 1. 数据集背景与营销映射

**处理对象**：真实电商营销场景的数据库 Schema（6 张表）。

| 数据表 | 记录数 | 核心字段 | 营销用途 |
|--------|--------|---------|---------|
| categories（类目） | 6 | category_id, category_name, parent_category_id | 商品分类管理 |
| customers（客户） | 200 | customer_id, name, phone, email, gender, customer_level, status | 客户画像、RFM分析 |
| products（商品） | 50 | product_id, product_name, category_id, brand, price, cost, stock | 商品管理、利润分析 |
| orders（订单） | 500 | order_id, customer_id, order_date, total_amount, status, channel | 消费行为、渠道分析 |
| order_items（订单明细） | ~1200 | item_id, order_id, product_id, quantity, unit_price, subtotal | 商品销量、GMV计算 |
| campaigns（营销活动） | 3 | campaign_id, campaign_name, channel, budget, target_audience | 营销活动管理 |

**Schema 设计要点**：
- **主键约束**：每表 PRIMARY KEY 保证唯一性
- **外键约束**：orders→customers, order_items→orders/products 保证参照完整性
- **CHECK 约束**：price > 0、stock >= 0、gender IN ('M','F','O') 保证数据有效性
- **GENERATED 列**：order_items.subtotal = quantity * unit_price（自动计算，防篡改）
- **DEFAULT 值**：customer_level DEFAULT '普通'、status DEFAULT 'active'

**数据来源**：基于艾瑞咨询电商用户画像分布（性别比/等级分布/渠道占比）用 numpy 生成，固定 seed=42 确保可复现。

**理论连接**：数据治理六维度（准确性/完整性/一致性/及时性/唯一性/有效性）是 AI 系统可靠性的前提。SQL 是关系型数据库的通用查询语言，从 Schema 设计（DDL）到数据查询（DQL）到数据治理（约束/索引/质量检查）形成完整闭环。


In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# 固定随机种子确保可复现 (可复现研究原则)
np.random.seed(42)

# ===== 电商营销数据库: 原始数据生成 =====
# 数据分布依据: 艾瑞咨询《中国电子商务研究报告》电商用户画像与消费分布
# 性别比 M:F:O ≈ 45:52:3; 客户等级 普通:银卡:金卡:钻石 ≈ 70:20:8:2

# --- 类目 (6个) ---
categories = [
    (1, '个护美妆', None),
    (2, '3C数码', None),
    (3, '食品饮料', None),
    (4, '服装鞋包', None),
    (5, '母婴用品', None),
    (6, '家居家纺', None),
]

# --- 客户 (200个) ---
surnames = ['王','李','张','刘','陈','杨','黄','赵','周','吴',
            '徐','孙','马','朱','胡','郭','林','何','高','罗']
givens = ['伟','芳','娜','敏','静','丽','强','磊','军','洋',
          '勇','艳','杰','娟','涛','明','超','霞','平','刚']
customer_levels = ['普通', '银卡', '金卡', '钻石']
level_probs = [0.70, 0.20, 0.08, 0.02]
cust_statuses = ['active', 'inactive', 'frozen']
cust_status_probs = [0.85, 0.12, 0.03]
genders = ['M', 'F', 'O']
gender_probs = [0.45, 0.52, 0.03]

customers = []
for i in range(1, 201):
    name = str(np.random.choice(surnames) + np.random.choice(givens))
    phone = '1' + str(np.random.choice(['3','5','7','8','9'])) + str(np.random.randint(100000000, 999999999))
    email = 'user{}@example.com'.format(i)
    gender = str(np.random.choice(genders, p=gender_probs))
    level = str(np.random.choice(customer_levels, p=level_probs))
    status = str(np.random.choice(cust_statuses, p=cust_status_probs))
    signup = '2024-{:02d}-{:02d}'.format(np.random.randint(1, 13), np.random.randint(1, 29))
    customers.append((i, name, phone, email, gender, level, status, signup))

# --- 商品 (50个) ---
# 价格区间依据各类目真实电商均价
product_pool = {
    1: ['保湿面霜','精华液','防晒霜','口红','面膜','洗面奶','香水','乳液','粉底液'],
    2: ['蓝牙耳机','智能手表','手机壳','充电宝','数据线','音箱','平板支架','键盘','鼠标'],
    3: ['坚果礼盒','咖啡豆','茶叶','巧克力','蛋白粉','维生素','蜂蜜','零食礼包'],
    4: ['T恤','牛仔裤','运动鞋','连衣裙','羽绒服','背包','帽子','袜子'],
    5: ['奶粉','纸尿裤','婴儿推车','玩具','绘本','奶瓶','婴儿洗护','安全座椅'],
    6: ['四件套','枕头','收纳盒','毛巾','台灯','拖把','餐具套装','地毯'],
}
brands = ['品牌A', '品牌B', '品牌C', '品牌D', '品牌E']
products = []
pid = 1
for cat_id in range(1, 7):
    for pname in product_pool[cat_id]:
        if cat_id in [1, 2]:
            base_price = np.random.randint(50, 1000)
        elif cat_id == 3:
            base_price = np.random.randint(20, 300)
        elif cat_id == 4:
            base_price = np.random.randint(50, 800)
        elif cat_id == 5:
            base_price = np.random.randint(30, 1500)
        else:
            base_price = np.random.randint(20, 500)
        cost = round(float(base_price) * float(np.random.uniform(0.4, 0.6)), 2)
        stock = int(np.random.randint(0, 500))
        brand = str(np.random.choice(brands))
        products.append((pid, str(pname), cat_id, brand, round(float(base_price), 2), cost, stock))
        pid += 1

# --- 订单 (500个) + 订单明细 ---
# 渠道分布: 微信35% 淘宝30% 抖音20% 京东15%
channels = ['wechat', 'taobao', 'douyin', 'jd']
channel_probs = [0.35, 0.30, 0.20, 0.15]
order_statuses = ['pending', 'paid', 'shipped', 'delivered', 'cancelled']
order_status_probs = [0.05, 0.10, 0.10, 0.70, 0.05]

orders = []
order_items = []
item_id = 1
for oid in range(1, 501):
    if oid <= 3:
        cid = 1  # 确保客户1有3个订单 (教学演示用)
    elif oid <= 10:
        cid = oid - 3  # 客户2-7各有1个订单
    else:
        cid = int(np.random.randint(1, 201))
    order_date = '2024-{:02d}-{:02d}'.format(np.random.randint(1, 13), np.random.randint(1, 29))
    channel = str(np.random.choice(channels, p=channel_probs))
    status = str(np.random.choice(order_statuses, p=order_status_probs))
    n_items = int(np.random.randint(1, 6))
    total = 0.0
    for _ in range(n_items):
        pid = int(np.random.randint(1, 51))
        qty = int(np.random.randint(1, 4))
        unit_price = products[pid - 1][4]
        total += qty * unit_price
        order_items.append((item_id, oid, pid, qty, unit_price))
        item_id += 1
    orders.append((oid, cid, order_date, round(total, 2), status, channel))

# --- 营销活动 (3个) ---
campaigns = [
    (1, '双11大促', 'taobao', 500000.0, '全量客户'),
    (2, '618年中购物节', 'jd', 300000.0, '银卡及以上'),
    (3, '抖音直播带货', 'douyin', 150000.0, '新客户'),
]

print("数据生成完成")
print("  类目: {} | 客户: {} | 商品: {} | 订单: {} | 明细: {} | 活动: {}".format(
    len(categories), len(customers), len(products), len(orders), len(order_items), len(campaigns)))


## TODO 1：用 sqlite3 创建电商营销数据库 Schema

**任务**：用 sqlite3 创建 6 张表（带主键/外键/CHECK/DEFAULT/GENERATED 约束），并插入数据。

**提示**：
- `sqlite3.connect(':memory:')` 创建内存数据库
- `conn.execute('PRAGMA foreign_keys = ON')` 启用外键约束（SQLite 默认关闭！）
- `cursor.execute('CREATE TABLE ...')` 建表，注意约束定义
- `GENERATED ALWAYS AS (expr) STORED` 定义自动计算列
- `cursor.executemany('INSERT ...', data)` 批量插入；GENERATED 列插入时不指定
- `conn.commit()` 提交事务

**理论连接**：DDL（Data Definition Language）定义数据库结构。好的 Schema 设计是数据治理的第一步--主键保证唯一性，外键保证参照完整性，CHECK 约束在入库时就拦截非法数据，GENERATED 列防篡改。

In [ ]:
# 1. 用 sqlite3 创建电商营销数据库 Schema + 插入数据
conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys = ON')   # 启用外键约束 (SQLite 默认关闭!)
cursor = conn.cursor()

# 建表: categories (类目, 自引用外键)
cursor.execute('''
CREATE TABLE categories (
    category_id    INTEGER PRIMARY KEY,
    category_name  TEXT NOT NULL,
    parent_category_id INTEGER,
    FOREIGN KEY (parent_category_id) REFERENCES categories(category_id)
)''')

# 建表: customers (客户, CHECK + DEFAULT)
cursor.execute('''
CREATE TABLE customers (
    customer_id    INTEGER PRIMARY KEY,
    name           TEXT NOT NULL,
    phone          TEXT,
    email          TEXT,
    gender         TEXT CHECK(gender IN ('M','F','O')),
    customer_level TEXT DEFAULT '普通',
    status         TEXT DEFAULT 'active',
    signup_date    TEXT
)''')

# 建表: products (商品, CHECK + 外键)
cursor.execute('''
CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category_id  INTEGER,
    brand        TEXT,
    price        REAL CHECK(price > 0),
    cost         REAL CHECK(cost >= 0),
    stock        INTEGER CHECK(stock >= 0),
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)''')

# 建表: orders (订单, CHECK + 外键)
cursor.execute('''
CREATE TABLE orders (
    order_id     INTEGER PRIMARY KEY,
    customer_id  INTEGER,
    order_date   TEXT,
    total_amount REAL CHECK(total_amount >= 0),
    status       TEXT CHECK(status IN ('pending','paid','shipped','delivered','cancelled')),
    channel      TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)''')

# 建表: order_items (订单明细, GENERATED 列 + 双外键)
cursor.execute('''
CREATE TABLE order_items (
    item_id    INTEGER PRIMARY KEY,
    order_id   INTEGER,
    product_id INTEGER,
    quantity   INTEGER CHECK(quantity > 0),
    unit_price REAL CHECK(unit_price > 0),
    subtotal   REAL GENERATED ALWAYS AS (quantity * unit_price) STORED,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
)''')

# 建表: campaigns (营销活动)
cursor.execute('''
CREATE TABLE campaigns (
    campaign_id    INTEGER PRIMARY KEY,
    campaign_name  TEXT NOT NULL,
    channel        TEXT,
    budget         REAL CHECK(budget >= 0),
    target_audience TEXT
)''')

# 插入数据
cursor.executemany('INSERT INTO categories VALUES (?,?,?)', categories)
cursor.executemany('INSERT INTO customers VALUES (?,?,?,?,?,?,?,?)', customers)
cursor.executemany('INSERT INTO products VALUES (?,?,?,?,?,?,?)', products)
cursor.executemany('INSERT INTO orders VALUES (?,?,?,?,?,?)', orders)
# GENERATED 列 subtotal 自动计算, INSERT 时不指定该列
cursor.executemany('INSERT INTO order_items (item_id, order_id, product_id, quantity, unit_price) VALUES (?,?,?,?,?)', order_items)
cursor.executemany('INSERT INTO campaigns VALUES (?,?,?,?,?)', campaigns)
conn.commit()

# 验证
for tbl in ['categories','customers','products','orders','order_items','campaigns']:
    cnt = cursor.execute('SELECT COUNT(*) FROM {}'.format(tbl)).fetchone()[0]
    print('  {}: {} 行'.format(tbl, cnt))

# 验证 GENERATED 列自动计算
sample = cursor.execute('SELECT item_id, quantity, unit_price, subtotal FROM order_items LIMIT 3').fetchall()
print('  GENERATED 列验证 (qty, price, subtotal):', sample)


## TODO 2：SELECT-WHERE-ORDER BY 基础查询

**任务**：用 SELECT-WHERE-ORDER BY 按品类/价格/时间筛选产品和订单，用 `pd.read_sql_query()` 转为 DataFrame。

**提示**：
- `JOIN ... ON ...` 连接 products 和 categories 获取类目名
- `WHERE` 筛选条件：`category_name = '个护美妆'`、`price BETWEEN 100 AND 500`、`order_date LIKE '2024-06%'`
- `ORDER BY price DESC` 按价格降序
- `pd.read_sql_query(sql, conn)` 执行 SELECT 返回 DataFrame

**营销场景**：筛选特定品类的产品做竞品分析，筛选特定价格区间做定价策略，筛选特定时段订单做促销效果评估。

In [ ]:
# 2. SELECT-WHERE-ORDER BY 基础查询
# 2a: 查询"个护美妆"类目的产品, 按价格降序
df2a = pd.read_sql_query('''
SELECT p.product_name, p.brand, p.price, p.stock, c.category_name
FROM products p
JOIN categories c ON p.category_id = c.category_id
WHERE c.category_name = '个护美妆'
ORDER BY p.price DESC
''', conn)
print('=== 2a 个护美妆产品 (按价格降序) ===')
print(df2a.to_string(index=False))

# 2b: 查询价格在 100-500 之间的所有产品
df2b = pd.read_sql_query('''
SELECT product_name, brand, price, stock
FROM products
WHERE price BETWEEN 100 AND 500
ORDER BY price
''', conn)
print('\n=== 2b 价格 100-500 产品 ({} 行) ==='.format(len(df2b)))
print(df2b.head(10).to_string(index=False))

# 2c: 查询 2024 年 6 月的已发货订单
df2c = pd.read_sql_query('''
SELECT order_id, customer_id, order_date, total_amount, channel
FROM orders
WHERE order_date LIKE '2024-06%' AND status = 'delivered'
ORDER BY order_date, total_amount DESC
''', conn)
print('\n=== 2c 2024年6月已发货订单 ({} 行) ==='.format(len(df2c)))
print(df2c.head(10).to_string(index=False))


## TODO 3：JOIN 多表连接 -- "某客户买了什么产品"

**任务**：用 JOIN 连接 customers-orders-order_items-products 四表，查询客户购买明细。

**提示**：
- **INNER JOIN**：`JOIN orders ON customers.customer_id = orders.customer_id` 只返回有匹配的行
- **LEFT JOIN**：`LEFT JOIN orders ON ...` 保留左表所有行（含未下单客户）
- 四表连接链：customers → orders → order_items → products
- `COALESCE(SUM(x), 0)` 将 NULL 转为 0
- `COUNT(DISTINCT o.order_id)` 统计不重复订单数

**营销场景**：客户360视图--从客户画像到购买明细的全链路查询，是推荐系统和客户分群的数据基础。

In [ ]:
# 3. JOIN 多表连接: 查"某客户买了什么产品"
# 3a: 查询客户ID=1 购买的所有产品 (四表连接)
df3a = pd.read_sql_query('''
SELECT cu.customer_id, cu.name, cu.customer_level,
       o.order_id, o.order_date, o.status,
       p.product_name, p.brand, p.category_id,
       oi.quantity, oi.unit_price, oi.subtotal
FROM customers cu
JOIN orders o       ON cu.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p     ON oi.product_id = p.product_id
WHERE cu.customer_id = 1
ORDER BY o.order_date
''', conn)
print('=== 3a 客户1 购买明细 ({} 条) ==='.format(len(df3a)))
print(df3a[['order_date','product_name','quantity','unit_price','subtotal']].to_string(index=False))
print('客户1 总消费: ¥{:.2f}'.format(df3a['subtotal'].sum()))

# 3b: LEFT JOIN 查询所有客户及订单数 (含未下单客户)
df3b = pd.read_sql_query('''
SELECT cu.customer_id, cu.name, cu.customer_level,
       COUNT(o.order_id) AS order_count,
       ROUND(COALESCE(SUM(o.total_amount), 0), 2) AS total_spending
FROM customers cu
LEFT JOIN orders o ON cu.customer_id = o.customer_id
GROUP BY cu.customer_id
ORDER BY total_spending DESC
LIMIT 15
''', conn)
print('\n=== 3b 客户消费排行 TOP15 (LEFT JOIN 含未下单) ===')
print(df3b.to_string(index=False))

# 未下单客户数
no_order = pd.read_sql_query('''
SELECT COUNT(*) AS no_order_count
FROM customers cu
LEFT JOIN orders o ON cu.customer_id = o.customer_id
WHERE o.order_id IS NULL
''', conn)
print('未下单客户数: {}'.format(no_order.iloc[0, 0]))


## TODO 4：GROUP BY-HAVING 聚合分析 -- 按品类/月份聚合 GMV

**任务**：用 GROUP BY 按品类和月份聚合销量 GMV，用 HAVING 筛选高 GMV 品类。

**提示**：
- `GROUP BY c.category_name` 按品类分组
- `SUM(oi.subtotal)` 计算 GMV（Gross Merchandise Volume）
- `COUNT(DISTINCT oi.order_id)` 统计不重复订单数
- `HAVING SUM(oi.subtotal) > 10000` 筛选高 GMV 品类（WHERE 在分组前过滤，HAVING 在分组后过滤）
- `substr(order_date, 1, 7)` 提取年月（如 '2024-06'）

**营销场景**：GMV 是电商核心指标。按品类聚合找 TOP 品类，按月份聚合看销售趋势，HAVING 筛选重点关注的高价值品类。

In [ ]:
# 4. GROUP BY-HAVING 聚合分析: 按品类/月份聚合 GMV
# 4a: 按品类聚合订单数/销量/GMV
df4a = pd.read_sql_query('''
SELECT c.category_name,
       COUNT(DISTINCT oi.order_id) AS order_count,
       SUM(oi.quantity)            AS total_qty,
       ROUND(SUM(oi.subtotal), 2)  AS gmv
FROM order_items oi
JOIN orders o       ON oi.order_id = o.order_id
JOIN products p     ON oi.product_id = p.product_id
JOIN categories c   ON p.category_id = c.category_id
GROUP BY c.category_name
ORDER BY gmv DESC
''', conn)
print('=== 4a 各品类 GMV 排行 ===')
print(df4a.to_string(index=False))

# 4b: 按月份聚合 GMV
df4b = pd.read_sql_query('''
SELECT substr(order_date, 1, 7) AS month,
       COUNT(*)                  AS order_count,
       ROUND(SUM(total_amount), 2) AS monthly_gmv
FROM orders
WHERE status != 'cancelled'
GROUP BY substr(order_date, 1, 7)
ORDER BY month
''', conn)
print('\n=== 4b 月度 GMV ===')
print(df4b.to_string(index=False))

# 4c: HAVING 筛选 GMV 超过 10000 的品类
df4c = pd.read_sql_query('''
SELECT c.category_name,
       ROUND(SUM(oi.subtotal), 2) AS gmv
FROM order_items oi
JOIN products p   ON oi.product_id = p.product_id
JOIN categories c ON p.category_id = c.category_id
GROUP BY c.category_name
HAVING SUM(oi.subtotal) > 10000
ORDER BY gmv DESC
''', conn)
print('\n=== 4c GMV > 10000 的品类 (HAVING) ===')
print(df4c.to_string(index=False))


## TODO 5：窗口函数 + 子查询 -- RANK 排名 + 累计求和 + RFM 分群

**任务**：用窗口函数（RANK/SUM OVER）和 CTE + 子查询实现热销商品排名、月度累计 GMV、RFM 客户分群。

**提示**：
- `RANK() OVER (ORDER BY SUM(oi.subtotal) DESC)` 按收入排名热销商品
- `SUM(monthly_gmv) OVER (ORDER BY month)` 计算累计 GMV
- `WITH cte AS (...)` 定义公共表表达式（CTE）
- `NTILE(4) OVER (ORDER BY ...)` 将数据四等分（RFM 各维度打分 1-4）
- `julianday('2024-12-31') - julianday(MAX(order_date))` 计算最近购买距今天数（Recency）

**RFM 理论**：
- **R（Recency）**：最近一次购买距今天数 -- 越小越好
- **F（Frequency）**：购买次数 -- 越大越好
- **M（Monetary）**：消费总金额 -- 越大越好
- 用 NTILE(4) 对每个维度打分 1-4，综合 r_score + f_score + m_score 分群

**营销场景**：RFM 是营销领域最经典的客户分群方法，高价值客户做 VIP 运营，流失风险客户做召回。

In [ ]:
# 5. 窗口函数 + 子查询: RANK 排名 + 累计求和 + RFM 分群
# 5a: RANK 排名热销商品 (按收入)
df5a = pd.read_sql_query('''
SELECT p.product_name, c.category_name,
       SUM(oi.quantity)           AS total_sold,
       ROUND(SUM(oi.subtotal), 2) AS total_revenue,
       RANK() OVER (ORDER BY SUM(oi.subtotal) DESC) AS revenue_rank
FROM order_items oi
JOIN products p   ON oi.product_id = p.product_id
JOIN categories c ON p.category_id = c.category_id
GROUP BY p.product_id
ORDER BY revenue_rank
LIMIT 10
''', conn)
print('=== 5a 热销商品 TOP10 (RANK) ===')
print(df5a.to_string(index=False))

# 5b: 累计求和: 按月累计 GMV
df5b = pd.read_sql_query('''
SELECT month, monthly_gmv,
       ROUND(SUM(monthly_gmv) OVER (ORDER BY month), 2) AS cumulative_gmv
FROM (
    SELECT substr(order_date, 1, 7) AS month,
           ROUND(SUM(total_amount), 2) AS monthly_gmv
    FROM orders
    WHERE status != 'cancelled'
    GROUP BY substr(order_date, 1, 7)
)
ORDER BY month
''', conn)
print('\n=== 5b 月度累计 GMV (窗口函数) ===')
print(df5b.to_string(index=False))

# 5c: RFM 客户分群 (CTE + NTILE 窗口函数 + CASE)
# R=Recency 最近购买距参考日天数; F=Frequency 购买次数; M=Monetary 消费总额
df5c = pd.read_sql_query('''
WITH rfm_base AS (
    SELECT o.customer_id,
           CAST(julianday('2024-12-31') - julianday(MAX(o.order_date)) AS INTEGER) AS recency,
           COUNT(DISTINCT o.order_id)  AS frequency,
           ROUND(SUM(o.total_amount), 2) AS monetary
    FROM orders o
    WHERE o.status != 'cancelled'
    GROUP BY o.customer_id
),
rfm_scored AS (
    SELECT customer_id, recency, frequency, monetary,
           NTILE(4) OVER (ORDER BY recency ASC)   AS r_score,
           NTILE(4) OVER (ORDER BY frequency DESC) AS f_score,
           NTILE(4) OVER (ORDER BY monetary DESC)  AS m_score
    FROM rfm_base
)
SELECT customer_id, recency, frequency, monetary,
       r_score, f_score, m_score,
       (r_score + f_score + m_score) AS rfm_total,
       CASE
           WHEN r_score >= 3 AND f_score >= 3 AND m_score >= 3 THEN '高价值客户'
           WHEN r_score >= 2 AND f_score >= 2 THEN '中等价值客户'
           WHEN r_score >= 2 THEN '低价值客户'
           ELSE '流失风险客户'
       END AS segment
FROM rfm_scored
ORDER BY rfm_total DESC
''', conn)
print('\n=== 5c RFM 客户分群 ===')
print(df5c.head(15).to_string(index=False))
print('\n各分群人数:')
print(df5c['segment'].value_counts().to_string())


## TODO 6：数据治理实操 -- 索引/缺失值/重复/数据质量六维度

**任务**：创建索引提升性能，检测缺失值和重复行，执行数据质量六维度审计，评估范式化。

**提示**：
- `CREATE INDEX idx_name ON table(col)` 创建索引
- `EXPLAIN QUERY PLAN SELECT ...` 查看查询计划是否用索引
- `SUM(CASE WHEN col IS NULL THEN 1 ELSE 0 END)` 统计缺失值
- `GROUP BY ... HAVING COUNT(*) > 1` 检测重复
- 数据质量六维度：准确性/完整性/一致性/及时性/唯一性/有效性

**数据治理视角**：索引提升查询性能（空间换时间），约束保障数据完整性（事前预防），质量审计发现问题（事后检查）。这是 DAMA-DMBOK 数据治理框架的执行层。

In [ ]:
# 6. 数据治理: 索引/缺失值/重复/数据质量六维度
# 6a: 创建索引提升查询性能
cursor.execute('CREATE INDEX idx_orders_customer ON orders(customer_id)')
cursor.execute('CREATE INDEX idx_orders_date ON orders(order_date)')
cursor.execute('CREATE INDEX idx_order_items_order ON order_items(order_id)')
cursor.execute('CREATE INDEX idx_order_items_product ON order_items(product_id)')
conn.commit()
print('=== 6a 索引创建完成 ===')

# EXPLAIN QUERY PLAN 验证索引使用
plan = cursor.execute(
    'EXPLAIN QUERY PLAN SELECT * FROM orders WHERE customer_id = 1'
).fetchall()
print('索引使用验证:', plan)

# 6b: 缺失值检测
df_missing = pd.read_sql_query('''
SELECT
    SUM(CASE WHEN name IS NULL OR name = '' THEN 1 ELSE 0 END)  AS missing_name,
    SUM(CASE WHEN phone IS NULL OR phone = '' THEN 1 ELSE 0 END) AS missing_phone,
    SUM(CASE WHEN email IS NULL OR email = '' THEN 1 ELSE 0 END) AS missing_email,
    SUM(CASE WHEN gender IS NULL THEN 1 ELSE 0 END)             AS missing_gender
FROM customers
''', conn)
print('\n=== 6b 客户表缺失值检测 ===')
print(df_missing.to_string(index=False))

# 6c: 重复行检测
df_dup_phone = pd.read_sql_query('''
SELECT phone, COUNT(*) AS cnt
FROM customers
WHERE phone IS NOT NULL
GROUP BY phone
HAVING COUNT(*) > 1
''', conn)
print('\n=== 6c 重复手机号检测 ({} 条) ==='.format(len(df_dup_phone)))
print(df_dup_phone.head(5).to_string(index=False))

df_dup_order = pd.read_sql_query('''
SELECT customer_id, order_date, COUNT(*) AS cnt
FROM orders
GROUP BY customer_id, order_date
HAVING COUNT(*) > 1
LIMIT 5
''', conn)
print('同客户同日重复订单 (前5):', len(df_dup_order), '组')

# 6d: 数据质量六维度检查
print('\n=== 6d 数据质量六维度审计 ===')

# 维度1 准确性: CHECK 约束已保障 price>0, total_amount>=0
acc = cursor.execute('SELECT COUNT(*) FROM products WHERE price <= 0').fetchone()[0]
print('1.准确性 (price<=0 的异常商品数):', acc)

# 维度2 完整性: 关键字段非空比例
df_comp = pd.read_sql_query('''
SELECT
    ROUND(AVG(CASE WHEN phone IS NOT NULL AND phone != '' THEN 1.0 ELSE 0 END) * 100, 2) AS phone_pct,
    ROUND(AVG(CASE WHEN email IS NOT NULL AND email != '' THEN 1.0 ELSE 0 END) * 100, 2) AS email_pct,
    ROUND(AVG(CASE WHEN gender IS NOT NULL THEN 1.0 ELSE 0 END) * 100, 2) AS gender_pct
FROM customers
''', conn)
print('2.完整性 (phone/email/gender 非空率%):')
print(df_comp.to_string(index=False))

# 维度3 一致性: order_items.unit_price 与 products.price 是否一致
df_cons = pd.read_sql_query('''
SELECT COUNT(*) AS inconsistent_count
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
WHERE ABS(oi.unit_price - p.price) > 0.01
''', conn)
print('3.一致性 (order_items 与 products 价格不一致数):', df_cons.iloc[0, 0])

# 维度4 及时性: 数据时间范围
df_time = pd.read_sql_query('''
SELECT MIN(order_date) AS earliest, MAX(order_date) AS latest
FROM orders
''', conn)
print('4.及时性 (订单时间范围):', df_time.iloc[0, 0], '~', df_time.iloc[0, 1])

# 维度5 唯一性: 主键唯一 + 重复检测
pk_check = cursor.execute(
    'SELECT COUNT(*) - COUNT(DISTINCT customer_id) FROM customers'
).fetchone()[0]
print('5.唯一性 (客户主键重复数):', pk_check, '| 重复手机号组数:', len(df_dup_phone))

# 维度6 有效性: 枚举值校验
df_valid = pd.read_sql_query('''
SELECT
    SUM(CASE WHEN gender NOT IN ('M','F','O') THEN 1 ELSE 0 END) AS invalid_gender,
    SUM(CASE WHEN status NOT IN ('active','inactive','frozen') THEN 1 ELSE 0 END) AS invalid_status
FROM customers
''', conn)
print('6.有效性 (无效 gender/status 数):', int(df_valid.iloc[0, 0]), '/', int(df_valid.iloc[0, 1]))

# 6e: 范式化评估
print('\n=== 6e 范式化评估 ===')
print('1NF: 每字段不可再分 -- 通过 (无数组/嵌套字段)')
print('2NF: 非主键完全依赖主键 -- 通过 (order_items.subtotal 依赖 item_id)')
print('3NF: 无传递依赖 -- 通过 (客户等级存 customers, 不冗余存 orders)')
print('反范式: orders.total_amount 冗余存于 orders 而非每次聚合 order_items')
print('  -> 适度反范式, 用空间换查询性能 (企业常见权衡)')


## 3. 反思与前沿

### 反思问题
1. 你的电商营销数据库 Schema 遵循了哪些范式化原则？orders.total_amount 是否是反范式设计？
2. RFM 分群后各层级客户人数如何？高价值客户占比多少？
3. 数据质量六维度检查发现了哪些问题？如何改进？
4. GENERATED 列和 CHECK 约束如何从源头保障数据质量？

### 2026 前沿：DAMA-DMBOK + Apache Iceberg + 湖仓一体

**DAMA-DMBOK 数据治理框架**：DAMA International 定义了数据管理的 11 个知识领域，数据治理是核心统筹领域。在 AI 营销系统中，DAMA-DMBOK 提供系统化的数据治理 checklist，是 AI 可靠性的制度保障。

**Apache Iceberg 开放表格式**：为数据湖提供 ACID 事务、时间旅行、Schema 演化能力，是"湖仓一体"的技术基石。Snowflake、Databricks、Trino、Spark 均已支持。在 AI 营销场景中，Iceberg 让你可以在同一份数据上同时做 BI 报表和 ML 训练。

**Great Expectations 数据质量监控**：声明式数据质量检查工具，用 Python/SQL 声明规则（如"customer_id 必须唯一""price 必须 > 0"），每次入库自动运行检查，将数据治理从"事后审计"升级为"事前预防"。

**数据仓库 vs 数据湖 vs 湖仓一体**：结构化订单存数据仓库（Schema-on-Write），非结构化客服对话存数据湖（Schema-on-Read），湖仓一体用 Iceberg/Delta Lake 统一两者。

> 参考阅读见 [reading.md](./reading.md) 的 DAMA-DMBOK / Apache Iceberg / Great Expectations 条目。